## Stage 2 — Patching-as-Instrument at L11 H16 (the causal axis of the fidelity map)

Follow-up to findings 05-06: head **L11 H16** is significant in all 6 types as a fixed
location (RSA, correlational). This notebook tests its CAUSAL side:

> If group A's L11 H16 activation is **grafted** with the activation belonging to
> group B (exactly the same prompt, only a different demographic sentence), does the
> model's predicted opinion distribution **move toward group B's REAL distribution** (not
> merely change at all)?

Scoring is always against the real survey `responses` (Wasserstein) — consistent with the whole
project: patching here is an **instrument for measuring fidelity**, not a steering tool
(the difference from llm-opinions: they use patching to shift the output, we
use it to measure whether that shift is demographically CORRECT).

**Conditions per (pair A→B, question):**

| condition | what is grafted | its role |
|---|---|---|
| `baseline` | — | the original predictions of A and B |
| `patch_L11H16` | the star head | main hypothesis |
| `patch_L18H14` | cross-type runner-up | comparison |
| `patch_top3` | L11H16 + L11H19 + L18H14 | combined effect |
| `patch_random` | 1 random head (fixed, outside the top list) | negative control |
| `self_patch` | A's own L11 H16 (subset) | mechanism sanity check (must be ~= baseline) |

Honest expectation: grafting 1 head out of 1024, at 1 token position, is a small lever —
the effect may be small. What matters is its **direction and consistency** against the random control
(paired Wilcoxon test), not its raw magnitude.


## Before running: Kaggle setup

1. **Accelerator**: GPU T4 x2. **Internet: On**.
2. **Attach dataset** `opinionqa_intersectional.csv` (same as notebook 07/08/09).
3. A session left over from a crash -> **RESTART SESSION** first.
4. **WHEN DONE, download** from `/kaggle/working/stage2_patching/`:
   `patching_rows.csv` (one row per experiment) + `patching_summary.csv` — both are
   small, so follow-up analysis can be done locally.

Estimate: model load ~10 min; ~7,000 short forward passes ~45-75 min. Total ~1.5 hours.


In [ ]:
!pip install -q -U "transformers>=4.44" accelerate scipy tqdm

In [ ]:
import os, sys, gc, glob, ast
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import wasserstein_distance, wilcoxon
from tqdm.auto import tqdm

sys.last_traceback = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for d in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(d)
        print(f"GPU {d}: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
        if free / total < 0.9:
            print(f"  WARNING: GPU {d} is not empty -> RESTART SESSION first!")


In [ ]:
MODEL_PATH = "mistralai/Mistral-7B-v0.1"

_candidates = glob.glob("/kaggle/input/**/opinionqa_intersectional.csv", recursive=True)
if _candidates:
    DATA_PATH = _candidates[0]
elif os.path.exists("opinionqa_intersectional.csv"):
    DATA_PATH = "opinionqa_intersectional.csv"
else:
    raise FileNotFoundError("opinionqa_intersectional.csv not found.")
print("Data:", DATA_PATH)

RANDOM_SEED = 42
TYPES_RUN = ["AGExPOLPARTY", "RELIGxPOLPARTY", "RACExRELIG"]  # strongest / biggest gain / weakest
N_PAIRS = 15        # directed pairs (A->B) per type
N_QUESTIONS = 25    # shared questions per type
MAX_OPTIONS = 6     # drop questions with more than 6 options (letters A-F)
N_SELF_PATCH = 3    # first pairs for the self-patch sanity check

STAR = (11, 16)             # star head (finding 06)
RUNNER = (18, 14)
TOP3 = [(11, 16), (11, 19), (18, 14)]

OUT_DIR = "/kaggle/working/stage2_patching"
os.makedirs(OUT_DIR, exist_ok=True)


## 1. Data: pick types, cell pairs, and questions per pair

Questions are chosen **per pair** (the intersection of questions answered by A AND B) —
coverage across cells is sparse; there are types (RACExRELIG) that do not have a single
question answered by every cell. Candidate pairs are required to share >= 15
questions. The last `options` entry "Refused" is dropped (length of `responses` = length of
`ordinal` < length of `options`).


In [ ]:
df = pd.read_csv(DATA_PATH)
for c in ["responses", "ordinal", "options"]:
    df[c] = df[c].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df["group_key"] = df["attribute"] + " :: " + df["group"]
df["n_opt"] = df["ordinal"].apply(len)

rng = np.random.default_rng(RANDOM_SEED)
plan = {}
MIN_SHARED_Q = 15
for ty in TYPES_RUN:
    sub = df[(df["attribute"] == ty) & (df["n_opt"] <= MAX_OPTIONS)]
    cells = sorted(sub["group_key"].unique().tolist())
    q_per_cell = sub.groupby("group_key")["qkey"].apply(set).to_dict()
    # questions are chosen PER PAIR (the intersection of A & B) — coverage across cells is sparse,
    # no question is answered by ALL cells in some types (e.g. RACExRELIG)
    all_pairs = [(a, b) for a in cells for b in cells
                 if a != b and len(q_per_cell[a] & q_per_cell[b]) >= MIN_SHARED_Q]
    pick = rng.choice(len(all_pairs), size=min(N_PAIRS, len(all_pairs)), replace=False)
    pairs = [all_pairs[k] for k in pick]
    pair_questions = {}
    for (a, b) in pairs:
        shared = sorted(q_per_cell[a] & q_per_cell[b])
        if len(shared) > N_QUESTIONS:
            shared = sorted(rng.choice(shared, size=N_QUESTIONS, replace=False).tolist())
        pair_questions[(a, b)] = shared
    cells_used = sorted({c for p in pairs for c in p})
    needed = sorted({(gk, qk) for (a, b), qs in pair_questions.items()
                     for qk in qs for gk in (a, b)})
    plan[ty] = dict(cells=cells_used, pairs=pairs, pair_questions=pair_questions, needed=needed)
    n_q = sum(len(v) for v in pair_questions.values())
    print(f"[{ty}] {len(cells_used)} cells, {len(pairs)} pairs (candidates {len(all_pairs)}), "
          f"{n_q} (pairs x questions), unique baselines {len(needed)}")

# lookup of question metadata & real distributions
qmeta = {}
real_resp = {}
for r in df.itertuples():
    qmeta[r.qkey] = (r.question, r.options[: r.n_opt], r.ordinal)
    real_resp[(r.group_key, r.qkey)] = np.array(r.responses, dtype=np.float64)


## 2. Prompt & readout position

The demographic sentence uses **template T0** (the bridge to the notebook 09 map — L11 H16
was verified strong across T0-T3). The prompt ends with `Answer:` -> prediction = the probability
distribution over the option letters at the next token (readout in SubPOP/llm-opinions style).
The patch is applied at that same last-token position.


In [ ]:
ATTR_LABELS = {
    "RACExRELIG":       ("race", "religion"),
    "RACExPOLPARTY":    ("race", "political party affiliation"),
    "RACExPOLIDEOLOGY": ("race", "political ideology"),
    "RELIGxPOLPARTY":   ("religion", "political party affiliation"),
    "EDUCATIONxINCOME": ("highest level of education", "household income"),
    "AGExPOLPARTY":     ("age group", "political party affiliation"),
}
LETTERS = ["A", "B", "C", "D", "E", "F"]

def demo_sentence(gk):
    ty, grp = gk.split(" :: ", 1)
    v1, v2 = grp.split(" | ", 1)
    l1, l2 = ATTR_LABELS[ty]
    return f"This survey respondent's {l1} is {v1} and their {l2} is {v2}."

def build_prompt(gk, qk):
    question, options, _ = qmeta[qk]
    lines = [demo_sentence(gk), "", f"Question: {question}"]
    for i, opt in enumerate(options):
        lines.append(f"{LETTERS[i]}) {opt}")
    lines.append("Answer:")
    return "\n".join(lines)

ty0 = TYPES_RUN[0]
gk0, qk0 = plan[ty0]["needed"][0]
print(build_prompt(gk0, qk0))


## 3. Load model, the patching machinery, and the distribution readout

In [ ]:
print(f"Loading {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="balanced", low_cpu_mem_usage=True
)
model.eval()
NUM_LAYERS = model.config.num_hidden_layers
NUM_HEADS = model.config.num_attention_heads
HEAD_DIM = model.config.hidden_size // NUM_HEADS

LETTER_IDS = [tokenizer.encode(f" {L}", add_special_tokens=False)[-1] for L in LETTERS]
assert len(set(LETTER_IDS)) == len(LETTER_IDS), "letter tokens are not unique!"
print("Letter token ids:", LETTER_IDS)

# a FIXED random control head (outside the top list of findings 05/06)
_forbidden = set(TOP3) | {(8, 21), (23, 13), (14, 1), (20, 14), (16, 2), (16, 10), (12, 28)}
rng_ctrl = np.random.default_rng(RANDOM_SEED + 7)
while True:
    RAND_HEAD = (int(rng_ctrl.integers(0, NUM_LAYERS)), int(rng_ctrl.integers(0, NUM_HEADS)))
    if RAND_HEAD not in _forbidden:
        break
print("Random control head:", RAND_HEAD)

CAPTURE_LAYERS = sorted({l for l, _ in TOP3} | {RAND_HEAD[0]})

# ---- hook machinery: capture donor + apply patch at the last token ----
_donor_capture = {}          # layer -> tensor [4096] (o_proj input at the last token)
_active_patch = {}           # (layer) -> list of (head, vec[128])

def _oproj_prehook(layer_idx):
    def fn(module, args):
        x = args[0]
        _donor_capture[layer_idx] = x[0, -1, :].detach().float().cpu()
        patches = _active_patch.get(layer_idx)
        if patches:
            x = x.clone()
            for h, vec in patches:
                x[0, -1, h * HEAD_DIM:(h + 1) * HEAD_DIM] = vec.to(x.device, x.dtype)
            return (x,) + tuple(args[1:])
        return None
    return fn

handles = [model.model.layers[L].self_attn.o_proj.register_forward_pre_hook(_oproj_prehook(L))
           for L in CAPTURE_LAYERS]
print(f"Hooks attached at layers {CAPTURE_LAYERS}")

@torch.no_grad()
def forward_pred(prompt, n_opt, patch_spec=None):
    """patch_spec: list of ((layer, head), donor_vec4096) -> letter distribution [n_opt]."""
    _active_patch.clear()
    if patch_spec:
        for (L, H), vec in patch_spec:
            seg = vec[H * HEAD_DIM:(H + 1) * HEAD_DIM]
            _active_patch.setdefault(L, []).append((H, seg))
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    logits = model(**inputs).logits[0, -1, :]
    _active_patch.clear()
    selected = logits[LETTER_IDS[:n_opt]].float()
    return torch.softmax(selected, dim=0).cpu().numpy()


## 4. Pass 1 — all baselines (cell x question): prediction + donor vector

Donor = the o_proj-input activation at the last token in layers 11 & 18 (and the random head's layer),
stored per (cell, question). Reused for every patch condition.


In [ ]:
baseline_pred = {}   # (gk, qk) -> dist
donors = {}          # (gk, qk) -> {layer: vec4096}

for ty in TYPES_RUN:
    p = plan[ty]
    for (gk, qk) in tqdm(p["needed"], desc=f"baseline {ty}"):
        if (gk, qk) in baseline_pred:
            continue
        n_opt = len(qmeta[qk][2])
        pred = forward_pred(build_prompt(gk, qk), n_opt)
        baseline_pred[(gk, qk)] = pred
        donors[(gk, qk)] = {L: _donor_capture[L].clone() for L in CAPTURE_LAYERS}
print(f"{len(baseline_pred)} baselines done.")


## 5. Pass 2 — patch conditions per pair A->B

Prompt = A's (A's demographic sentence + the question). Donor = B's, from
the SAME question. What is measured: the WD from the prediction to A's REAL and B's REAL distribution.


In [ ]:
def wd(pred, real, ordinal):
    return wasserstein_distance(ordinal, ordinal, u_weights=pred, v_weights=real)

CONDITIONS = {
    "patch_L11H16": [STAR],
    "patch_L18H14": [RUNNER],
    "patch_top3":   TOP3,
    "patch_random": [RAND_HEAD],
}

rows = []
for ty in TYPES_RUN:
    p = plan[ty]
    for pi, (A, B) in enumerate(tqdm(p["pairs"], desc=f"patch {ty}")):
        for qk in p["pair_questions"][(A, B)]:
            question, options, ordinal = qmeta[qk]
            n_opt = len(ordinal)
            realA, realB = real_resp[(A, qk)], real_resp[(B, qk)]
            predA, predB = baseline_pred[(A, qk)], baseline_pred[(B, qk)]
            prompt_A = build_prompt(A, qk)
            base = dict(attr_type=ty, pair=f"{A} -> {B}", qkey=qk,
                        wd_A_to_realA=wd(predA, realA, ordinal),
                        wd_A_to_realB=wd(predA, realB, ordinal),
                        wd_B_to_realB=wd(predB, realB, ordinal))
            for cond, heads in CONDITIONS.items():
                spec = [((L, H), donors[(B, qk)][L]) for (L, H) in heads]
                pp = forward_pred(prompt_A, n_opt, patch_spec=spec)
                rows.append(dict(base, condition=cond,
                                 wd_patch_to_realB=wd(pp, realB, ordinal),
                                 wd_patch_to_realA=wd(pp, realA, ordinal),
                                 wd_patch_to_predB=wd(pp, predB, ordinal)))
            if pi < N_SELF_PATCH:  # sanity: graft A with A's own activation
                spec = [(STAR, donors[(A, qk)][STAR[0]])]
                pp = forward_pred(prompt_A, n_opt, patch_spec=spec)
                rows.append(dict(base, condition="self_patch",
                                 wd_patch_to_realB=wd(pp, realB, ordinal),
                                 wd_patch_to_realA=wd(pp, realA, ordinal),
                                 wd_patch_to_predB=wd(pp, predB, ordinal)))

res = pd.DataFrame(rows)
res["shift_to_realB"] = res["wd_A_to_realB"] - res["wd_patch_to_realB"]   # + = moved closer to REAL B
res["shift_from_realA"] = res["wd_patch_to_realA"] - res["wd_A_to_realA"] # + = moved away from REAL A
res.to_csv(os.path.join(OUT_DIR, "patching_rows.csv"), index=False)
print(res.shape, "-> patching_rows.csv")


## 6. Recap + statistical test

`shift_to_realB` > 0 = A's prediction moved CLOSER to B's real distribution. Paired Wilcoxon
test: the main condition's shift vs the random control's shift, on the same
(pair x question).


In [ ]:
summary_rows = []
print("=" * 100)
for ty in TYPES_RUN:
    r_ty = res[res["attr_type"] == ty]
    rand = r_ty[r_ty["condition"] == "patch_random"].set_index(["pair", "qkey"])["shift_to_realB"]
    for cond in ["patch_L11H16", "patch_L18H14", "patch_top3", "patch_random", "self_patch"]:
        s = r_ty[r_ty["condition"] == cond].set_index(["pair", "qkey"])["shift_to_realB"]
        if len(s) == 0:
            continue
        mean_shift = float(s.mean())
        pos_rate = float((s > 0).mean())
        p_vs_rand = np.nan
        if cond not in ("patch_random", "self_patch"):
            joined = pd.concat([s, rand], axis=1, keys=["c", "r"]).dropna()
            if len(joined) > 10 and not np.allclose(joined["c"], joined["r"]):
                p_vs_rand = float(wilcoxon(joined["c"], joined["r"]).pvalue)
        summary_rows.append(dict(attr_type=ty, condition=cond, n=len(s),
                                 mean_shift_to_realB=mean_shift, pct_positive=pos_rate,
                                 p_wilcoxon_vs_random=p_vs_rand))
        p_str = "-" if np.isnan(p_vs_rand) else f"{p_vs_rand:.4f}"
        print(f"{ty:16s} {cond:14s} n={len(s):4d} | mean shift={mean_shift:+.4f} | "
              f">0: {pos_rate:.0%} | p vs random: {p_str}")

summary = pd.DataFrame(summary_rows)
summary.to_csv(os.path.join(OUT_DIR, "patching_summary.csv"), index=False)

# scale context: how far apart A and B really are, in reality & in prediction
print("\nWD scale context (baseline):")
for ty in TYPES_RUN:
    r_ty = res[res["attr_type"] == ty].drop_duplicates(subset=["pair", "qkey"])
    print(f"{ty:16s} WD(predA, realA)={r_ty['wd_A_to_realA'].mean():.4f}  "
          f"WD(predA, realB)={r_ty['wd_A_to_realB'].mean():.4f}  "
          f"WD(predB, realB)={r_ty['wd_B_to_realB'].mean():.4f}")


## How to read the results & checklist

- **The main hypothesis passes** if: `patch_L11H16` has a mean `shift_to_realB` > 0,
  a Wilcoxon p vs `patch_random` < 0.05, and `self_patch` ~ 0 (mechanism sanity).
- `patch_top3` > `patch_L11H16` = the identity signal is spread over several heads (not
  just one); `patch_random` ~ 0 = the change is not merely "disturb anything and it moves".
- Compare the shift magnitude against the scale context: `WD(predA, realB) - WD(predB, realB)`
  = the maximum distance that could possibly be covered. A small but consistent & significant shift =
  the instrument works; zero everywhere = there is no causal fidelity at this head
  (also a finding — correlational geometry != causal).
- The type hierarchy must be consistent with finding 06: AGE/RELIGxPP > RACExRELIG.

**Download:** `patching_rows.csv` + `patching_summary.csv` ->
`results/05_patching_v1/`. Results -> finding 07.
